# 문서 타입 분류 — 전체학습/추론/제출

**프로젝트 구조:**
```
project/
├── src/
|    ├── 00_eda.ipynb         # EDA
|    ├── 01_main.ipynb        # 실험 노트북 (K-FOLD, Split, 모델 비교)
|    ├── 02_main_total.ipynb  # 전체데이터 학습, 추론, 제출 
|    ├── config.py            # 하이퍼파라미터 & 경로 
|    ├── preprocess.py        # 오프라인 증강 함수
|    ├── augmentation.py      # 온라인 증강 (v1/v2/v3 전환) & TTA
|    ├── dataset.py           # Dataset, MixUp, CutMix
|    ├── train.py             # 학습/검증 함수
|    ├── inference.py         # TTA 추론, 앙상블 
├── data/
├── outputs/
├── requirements.txt
```

In [ ]:
import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

import warnings
warnings.filterwarnings('ignore')

import importlib
import config
importlib.reload(config)
from config import *
import train
importlib.reload(train)

import augmentation
importlib.reload(augmentation)
from augmentation import get_train_transforms
from dataset import DocDataset
from train import build_model, get_class_weights, train_one_epoch 
from inference import predict_tta, save_submission, ensemble_from_files

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG['seed'])
print(f'Device: {CFG["device"]}')
print(f'Model: {CFG["model_name"]} | Size: {CFG["img_size"]}')
print(f'use_mixup={CFG["use_mixup"]}, use_cutmix={CFG["use_cutmix"]}') # 실제 적용된 config값 확인

---
## 1. 데이터 로드

In [ ]:
df_train = pd.read_csv(TRAIN_CSV)
df_meta  = pd.read_csv(META_CSV)
df_test  = pd.read_csv(SAMPLE_CSV)
target_to_name = dict(zip(df_meta['target'], df_meta['class_name']))

print(f'Train: {len(df_train)}장 | Test: {len(df_test)}장')

class_counts = df_train['target'].value_counts().sort_index()
ratio = class_counts.max() / class_counts.min()
print(f'\n불균형 비율: {ratio:.2f}x (max={class_counts.max()} / min={class_counts.min()})')
print(class_counts)

---
## 2. 1번 모델 학습

In [ ]:
# DataLoader (전체 데이터 학습 — 최종 제출용)
train_ds = DocDataset(df_train, TRAIN_DIR,
                      get_train_transforms(CFG['img_size'], version='v3'))

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'],
                          shuffle=True, num_workers=CFG['num_workers'],
                          pin_memory=True, drop_last=True)

# 모델
model = build_model(CFG['model_name'], CFG['num_classes'],
                    CFG['pretrained']).to(CFG['device'])
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'{CFG["model_name"]}: {n_params:.1f}M params | {len(df_train)}장 전체 학습 (v3)')

# Loss
class_weights = get_class_weights(df_train, CFG['num_classes'], CFG['device'])
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Optimizer & Scheduler
optimizer = torch.optim.AdamW(model.parameters(),
                               lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG['epochs'], eta_min=1e-6)

In [ ]:
model_tag = CFG['model_name'].replace('.', '_').replace('/', '_')
history = {'loss': [], 'f1': []}

print('=' * 60)
print(f'{CFG["model_name"]} | {len(df_train)}장 전체 | {CFG["epochs"]}ep')
print('=' * 60)

for epoch in range(CFG['epochs']):
    loss, f1 = train_one_epoch(model, train_loader, criterion,
                                optimizer, scheduler, CFG)
    history['loss'].append(loss)
    history['f1'].append(f1)

    lr = optimizer.param_groups[0]['lr']
    print(f'  Epoch {epoch+1:2d}/{CFG["epochs"]} │ '
          f'Loss={loss:.4f} F1={f1:.4f} LR={lr:.2e}')

# 모델 저장
torch.save(model.state_dict(), OUTPUT_DIR / f'{model_tag}.pth')
print(f'\n모델 저장: {OUTPUT_DIR / f"{model_tag}.pth"}')

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(history['loss']); axes[0].set_title('Loss')
axes[1].plot(history['f1']);   axes[1].set_title('F1')
plt.suptitle(CFG['model_name'], fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# === 노이즈 라벨 탐지 ===
from augmentation import get_valid_transforms

check_ds = DocDataset(df_train, TRAIN_DIR, get_valid_transforms(CFG['img_size']))
check_loader = DataLoader(check_ds, batch_size=CFG['batch_size'],
                          shuffle=False, num_workers=CFG['num_workers'])

all_preds, all_probs = [], []
with torch.no_grad():
    for images, labels in tqdm(check_loader, desc='노이즈 탐지'):
        outputs = model(images.to(CFG['device']))
        probs = torch.softmax(outputs, dim=1)
        all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        all_probs.append(probs.cpu().numpy())

all_probs = np.concatenate(all_probs)
all_preds = np.array(all_preds)
true_labels = df_train['target'].values

wrong_mask = all_preds != true_labels
wrong_df = df_train[wrong_mask].copy()
wrong_df['pred'] = all_preds[wrong_mask]
wrong_df['confidence'] = all_probs[wrong_mask].max(axis=1)

print(f'오분류 샘플: {len(wrong_df)}개 / {len(df_train)}개')
for _, row in wrong_df.sort_values('confidence', ascending=False).iterrows():
    print(f'  ID={row["ID"]} | 라벨={row["target"]}({target_to_name[row["target"]]}) '
          f'→ 예측={row["pred"]}({target_to_name[row["pred"]]}) '
          f'| conf={row["confidence"]:.3f}')

---
## 3. 1번 모델 : 추론 + 제출

In [ ]:
predictions, probs = predict_tta(model, df_test, TEST_DIR, CFG)

np.save(OUTPUT_DIR / f'probs_{model_tag}.npy', probs)
save_submission(predictions, df_test, OUTPUT_DIR / f'submission_{model_tag}.csv')

max_p = probs.max(axis=1)
print(f'\nConfidence: mean={max_p.mean():.3f} | >0.9: {(max_p>0.9).sum()}장 | <0.5: {(max_p<0.5).sum()}장')
print(f'예측 분포:')
print(pd.Series(predictions).value_counts().sort_index())

del train_loader
torch.cuda.empty_cache()

---
## 4. 2번 모델 학습

In [ ]:
# 모델 선택 (한번에 하나만)
# ConvNeXt-Small 
CFG['model_name'] = 'convnext_small.fb_in22k_ft_in1k'
CFG['img_size'] = 384; CFG['lr'] = 1e-4; CFG['batch_size'] = 12

# # Swin-Small 
# CFG['model_name'] = 'swin_small_patch4_window7_224'
# CFG['img_size'] = 224; CFG['lr'] = 5e-5; CFG['batch_size'] = 32

# Swin-Base
# small -> base : batch_size(32 → 16, 파라미터가 2배라 메모리 더 씀)
# CFG['model_name'] = 'swin_base_patch4_window7_224'
# CFG['img_size'] = 224; CFG['lr'] = 5e-5; CFG['batch_size'] = 16 

# EfficientNetV2-S 
# CFG['model_name'] = 'tf_efficientnetv2_s.in21k_ft_in1k'
# CFG['img_size'] = 384; CFG['lr'] = 1e-4; CFG['batch_size'] = 16

In [ ]:
# DataLoader (전체 데이터 학습 — 최종 제출용)
train_ds = DocDataset(df_train, TRAIN_DIR,
                      get_train_transforms(CFG['img_size'], version='v2'))
train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'],
                          shuffle=True, num_workers=CFG['num_workers'],
                          pin_memory=True, drop_last=True)

# 모델
model = build_model(CFG['model_name'], CFG['num_classes'],
                    CFG['pretrained']).to(CFG['device'])
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'{CFG["model_name"]}: {n_params:.1f}M params | {len(df_train)}장 전체 학습')

# Loss
class_weights = get_class_weights(df_train, CFG['num_classes'], CFG['device'])
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Optimizer & Scheduler
optimizer = torch.optim.AdamW(model.parameters(),
                               lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG['epochs'], eta_min=1e-6)

In [ ]:
model_tag = CFG['model_name'].replace('.', '_').replace('/', '_')
history = {'loss': [], 'f1': []}

print('=' * 60)
print(f'{CFG["model_name"]} | {len(df_train)}장 전체 | {CFG["epochs"]}ep')
print('=' * 60)

for epoch in range(CFG['epochs']):
    loss, f1 = train_one_epoch(model, train_loader, criterion,
                                optimizer, scheduler, CFG)
    history['loss'].append(loss)
    history['f1'].append(f1)

    lr = optimizer.param_groups[0]['lr']
    print(f'  Epoch {epoch+1:2d}/{CFG["epochs"]} │ '
          f'Loss={loss:.4f} F1={f1:.4f} LR={lr:.2e}')

# 모델 저장
torch.save(model.state_dict(), OUTPUT_DIR / f'{model_tag}.pth')
print(f'\n모델 저장: {OUTPUT_DIR / f"{model_tag}.pth"}')

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(history['loss']); axes[0].set_title('Loss')
axes[1].plot(history['f1']);   axes[1].set_title('F1')
plt.suptitle(CFG['model_name'], fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# === 노이즈 라벨 탐지 ===
from augmentation import get_valid_transforms

check_ds = DocDataset(df_train, TRAIN_DIR, get_valid_transforms(CFG['img_size']))
check_loader = DataLoader(check_ds, batch_size=CFG['batch_size'],
                          shuffle=False, num_workers=CFG['num_workers'])

all_preds, all_probs = [], []
with torch.no_grad():
    for images, labels in tqdm(check_loader, desc='노이즈 탐지'):
        outputs = model(images.to(CFG['device']))
        probs = torch.softmax(outputs, dim=1)
        all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        all_probs.append(probs.cpu().numpy())

all_probs = np.concatenate(all_probs)
all_preds = np.array(all_preds)
true_labels = df_train['target'].values

wrong_mask = all_preds != true_labels
wrong_df = df_train[wrong_mask].copy()
wrong_df['pred'] = all_preds[wrong_mask]
wrong_df['confidence'] = all_probs[wrong_mask].max(axis=1)

print(f'오분류 샘플: {len(wrong_df)}개 / {len(df_train)}개')
for _, row in wrong_df.sort_values('confidence', ascending=False).iterrows():
    print(f'  ID={row["ID"]} | 라벨={row["target"]}({target_to_name[row["target"]]}) '
          f'→ 예측={row["pred"]}({target_to_name[row["pred"]]}) '
          f'| conf={row["confidence"]:.3f}')

---
## 5. 2번 모델: 추론 + 제출

In [ ]:
predictions, probs = predict_tta(model, df_test, TEST_DIR, CFG)

np.save(OUTPUT_DIR / f'probs_{model_tag}.npy', probs)
save_submission(predictions, df_test, OUTPUT_DIR / f'submission_{model_tag}.csv')

max_p = probs.max(axis=1)
print(f'\nConfidence: mean={max_p.mean():.3f} | >0.9: {(max_p>0.9).sum()}장 | <0.5: {(max_p<0.5).sum()}장')
print(f'예측 분포:')
print(pd.Series(predictions).value_counts().sort_index())

del train_loader
torch.cuda.empty_cache()

---
## 6. 앙상블

In [ ]:
ensemble_preds, ensemble_probs = ensemble_from_files([
    'probs_efficientnet_b3.npy',
    'probs_convnext_small_fb_in22k_ft_in1k.npy',
    # 'probs_swin_small_patch4_window7_224.npy',
], df_test, OUTPUT_DIR)

---
## 7. 최적화: TTA 5개 -> 6개

In [ ]:
import importlib, augmentation, inference
importlib.reload(augmentation)
importlib.reload(inference)
from inference import predict_tta, save_submission, ensemble_from_files
from augmentation import get_tta_transforms
print(f'TTA: {len(get_tta_transforms(384))}개')

In [ ]:
# 1번 모델: B3 추론
CFG['model_name'] = 'efficientnet_b3'
CFG['img_size'] = 384; CFG['batch_size'] = 16
model_tag = 'efficientnet_b3'

model = build_model('efficientnet_b3', 17, pretrained=False).to(CFG['device'])
model.load_state_dict(torch.load(OUTPUT_DIR / f'{model_tag}.pth', map_location=CFG['device']))
model.eval()

predictions, probs = predict_tta(model, df_test, TEST_DIR, CFG)
np.save(OUTPUT_DIR / f'probs_{model_tag}_tta6.npy', probs)
save_submission(predictions, df_test, OUTPUT_DIR / f'submission_{model_tag}_tta6.csv')
print(f'Confidence: mean={probs.max(axis=1).mean():.3f}')

del model; torch.cuda.empty_cache()

In [ ]:
# 2번 모델: ConvNeXt 추론
CFG['model_name'] = 'convnext_small.fb_in22k_ft_in1k'
CFG['img_size'] = 384; CFG['batch_size'] = 12
model_tag = 'convnext_small_fb_in22k_ft_in1k'

model = build_model('convnext_small.fb_in22k_ft_in1k', 17, pretrained=False).to(CFG['device'])
model.load_state_dict(torch.load(OUTPUT_DIR / f'{model_tag}.pth', map_location=CFG['device']))
model.eval()

predictions, probs = predict_tta(model, df_test, TEST_DIR, CFG)
np.save(OUTPUT_DIR / f'probs_{model_tag}_tta6.npy', probs)
save_submission(predictions, df_test, OUTPUT_DIR / f'submission_{model_tag}_tta6.csv')
print(f'Confidence: mean={probs.max(axis=1).mean():.3f}')

del model; torch.cuda.empty_cache()

In [ ]:
# 앙상블
ensemble_preds, ensemble_probs = ensemble_from_files([
    'probs_efficientnet_b3_tta6.npy',
    'probs_convnext_small_fb_in22k_ft_in1k_tta6.npy',
], df_test, OUTPUT_DIR, save_name='submission_ensemble_tta6.csv')

---
## 8. 최적화: TTA 6개 -> 8개

In [ ]:
import os, random, importlib
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
import warnings; warnings.filterwarnings('ignore')

import config; importlib.reload(config); from config import *
CFG['use_wandb'] = False
import augmentation; importlib.reload(augmentation)
import inference; importlib.reload(inference)
from augmentation import get_tta_transforms
from dataset import DocDataset
from train import build_model
from inference import predict_tta, save_submission, ensemble_from_files

df_test = pd.read_csv(SAMPLE_CSV)
print(f'TTA: {len(get_tta_transforms(384))}개')

In [ ]:
CFG['img_size'] = 384; CFG['batch_size'] = 16; CFG['use_tta'] = True
model = build_model('efficientnet_b3', 17, pretrained=False).to(CFG['device'])
model.load_state_dict(torch.load(OUTPUT_DIR / 'efficientnet_b3.pth', map_location=CFG['device']))
model.eval()

predictions, probs = predict_tta(model, df_test, TEST_DIR, CFG)
np.save(OUTPUT_DIR / 'probs_efficientnet_b3_tta8.npy', probs)
save_submission(predictions, df_test, OUTPUT_DIR / 'submission_efficientnet_b3_tta8.csv')
print(f'Confidence: mean={probs.max(axis=1).mean():.3f}')

del model; torch.cuda.empty_cache()

In [ ]:
CFG['img_size'] = 384; CFG['batch_size'] = 12; CFG['use_tta'] = True
model = build_model('convnext_small.fb_in22k_ft_in1k', 17, pretrained=False).to(CFG['device'])
model.load_state_dict(torch.load(OUTPUT_DIR / 'convnext_small_fb_in22k_ft_in1k.pth', map_location=CFG['device']))
model.eval()

predictions, probs = predict_tta(model, df_test, TEST_DIR, CFG)
np.save(OUTPUT_DIR / 'probs_convnext_small_fb_in22k_ft_in1k_tta8.npy', probs)
save_submission(predictions, df_test, OUTPUT_DIR / 'submission_convnext_small_fb_in22k_ft_in1k_tta8.csv')
print(f'Confidence: mean={probs.max(axis=1).mean():.3f}')

del model; torch.cuda.empty_cache()

In [ ]:
ensemble_preds, ensemble_probs = ensemble_from_files([
    'probs_efficientnet_b3_tta8.npy',
    'probs_convnext_small_fb_in22k_ft_in1k_tta8.npy',
], df_test, OUTPUT_DIR, save_name='submission_ensemble_tta8.csv')

--
## 9. 최적화: TTA 8개 -> 10개

In [ ]:
import os, random, importlib
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
import warnings; warnings.filterwarnings('ignore')

import config; importlib.reload(config); from config import *
CFG['use_wandb'] = False
import augmentation; importlib.reload(augmentation)
import inference; importlib.reload(inference)
from augmentation import get_tta_transforms
from dataset import DocDataset
from train import build_model
from inference import predict_tta, save_submission, ensemble_from_files

df_test = pd.read_csv(SAMPLE_CSV)
print(f'TTA: {len(get_tta_transforms(384))}개')

In [ ]:
CFG['img_size'] = 384; CFG['batch_size'] = 16; CFG['use_tta'] = True
model = build_model('efficientnet_b3', 17, pretrained=False).to(CFG['device'])
model.load_state_dict(torch.load(OUTPUT_DIR / 'efficientnet_b3.pth', map_location=CFG['device']))
model.eval()

predictions, probs = predict_tta(model, df_test, TEST_DIR, CFG)
np.save(OUTPUT_DIR / 'probs_efficientnet_b3_tta10.npy', probs)
save_submission(predictions, df_test, OUTPUT_DIR / 'submission_efficientnet_b3_tta10.csv')
print(f'Confidence: mean={probs.max(axis=1).mean():.3f}')

del model; torch.cuda.empty_cache()

In [ ]:
CFG['img_size'] = 384; CFG['batch_size'] = 12; CFG['use_tta'] = True
model = build_model('convnext_small.fb_in22k_ft_in1k', 17, pretrained=False).to(CFG['device'])
model.load_state_dict(torch.load(OUTPUT_DIR / 'convnext_small_fb_in22k_ft_in1k.pth', map_location=CFG['device']))
model.eval()

predictions, probs = predict_tta(model, df_test, TEST_DIR, CFG)
np.save(OUTPUT_DIR / 'probs_convnext_small_fb_in22k_ft_in1k_tta10.npy', probs)
save_submission(predictions, df_test, OUTPUT_DIR / 'submission_convnext_small_fb_in22k_ft_in1k_tta10.csv')
print(f'Confidence: mean={probs.max(axis=1).mean():.3f}')

del model; torch.cuda.empty_cache()

In [ ]:
ensemble_preds, ensemble_probs = ensemble_from_files([
    'probs_efficientnet_b3_tta10.npy',
    'probs_convnext_small_fb_in22k_ft_in1k_tta10.npy',
], df_test, OUTPUT_DIR, save_name='submission_ensemble_tta10.csv')

---
## 10. 최적화: soft voting

In [ ]:
import numpy as np
import pandas as pd
from config import *

b3_probs = np.load(OUTPUT_DIR / 'probs_efficientnet_b3_tta10.npy')
conv_probs = np.load(OUTPUT_DIR / 'probs_convnext_small_fb_in22k_ft_in1k_tta10.npy')
df_test = pd.read_csv(SAMPLE_CSV)

weights_list = [
    (0.50, 0.50),  # baseline
    (0.45, 0.55),
    (0.40, 0.60),
    (0.35, 0.65),
    (0.30, 0.70),
]

for w_b3, w_conv in weights_list:
    ens = w_b3 * b3_probs + w_conv * conv_probs
    preds = np.argmax(ens, axis=1)
    fname = f'submission_w{int(w_b3*100)}_{int(w_conv*100)}.csv'
    df_test['target'] = preds
    df_test.to_csv(OUTPUT_DIR / fname, index=False)
    
    # 예측 분포 차이 확인
    baseline_ens = 0.5 * b3_probs + 0.5 * conv_probs
    diff = (preds != np.argmax(baseline_ens, axis=1)).sum()
    print(f'  B3={w_b3:.2f} | ConvNeXt={w_conv:.2f} → {fname} | baseline과 {diff}장 차이')

print(f'\n{len(weights_list)}개 CSV 저장 완료')

---
## 11. 최적화: 클래스별 weighted ensemble

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedShuffleSplit
from config import *

# probs 로드
b3_probs = np.load(OUTPUT_DIR / 'probs_efficientnet_b3_tta10.npy')
conv_probs = np.load(OUTPUT_DIR / 'probs_convnext_small_fb_in22k_ft_in1k_tta10.npy')
df_test = pd.read_csv(SAMPLE_CSV)

# 9:1 val의 _best.pth로 혼동 클래스 분석
from augmentation import get_valid_transforms
from dataset import DocDataset
from train import build_model
from torch.utils.data import DataLoader
import torch

df_train = pd.read_csv(TRAIN_CSV)
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
_, val_idx = next(sss.split(df_train, df_train['target']))
df_val = df_train.iloc[val_idx]
val_labels = df_val['target'].values

# 각 모델의 val probs (clean)
val_probs = {}
for name, tag, img_size, bs in [
    ('efficientnet_b3', 'efficientnet_b3', 384, 16),
    ('convnext_small.fb_in22k_ft_in1k', 'convnext_small_fb_in22k_ft_in1k', 384, 12),
]:
    model = build_model(name, 17, pretrained=False).to('cuda')
    model.load_state_dict(torch.load(OUTPUT_DIR / f'{tag}_best.pth', map_location='cuda'))
    model.eval()
    ds = DocDataset(df_val, TRAIN_DIR, get_valid_transforms(img_size))
    loader = DataLoader(ds, batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)
    p = []
    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images.to('cuda'))
            p.append(torch.softmax(outputs, dim=1).cpu().numpy())
    val_probs[tag] = np.concatenate(p, axis=0)
    del model; torch.cuda.empty_cache()

# 클래스별 정확도 비교
print(f'{"클래스":<8} {"B3 정확도":<12} {"ConvNeXt 정확도":<15} {"더 강한 모델"}')
print(f'{"─"*50}')
b3_preds = np.argmax(val_probs['efficientnet_b3'], axis=1)
conv_preds = np.argmax(val_probs['convnext_small_fb_in22k_ft_in1k'], axis=1)

class_strength = {}  # 클래스별 어느 모델이 강한지
for c in range(17):
    mask = val_labels == c
    b3_acc = (b3_preds[mask] == c).mean()
    conv_acc = (conv_preds[mask] == c).mean()
    stronger = 'ConvNeXt' if conv_acc > b3_acc else 'B3' if b3_acc > conv_acc else 'same'
    class_strength[c] = stronger
    if b3_acc != conv_acc:
        print(f'  {c:<6} {b3_acc:<12.3f} {conv_acc:<15.3f} {stronger}')

# 클래스별 가중치 자동 설정
print(f'\n{"="*50}')
print(f'클래스별 Weighted Ensemble 실험')
print(f'{"="*50}')

# 기본 40:60 + 혼동 클래스 보정
configs = [
    ('baseline 40:60', {c: (0.4, 0.6) for c in range(17)}),
    ('클래스별 최적화 (강한쪽 +0.1)',
     {c: (0.3, 0.7) if class_strength[c] == 'ConvNeXt'
         else (0.5, 0.5) if class_strength[c] == 'B3'
         else (0.4, 0.6)
      for c in range(17)}),
    ('클래스별 최적화 (강한쪽 +0.15)',
     {c: (0.25, 0.75) if class_strength[c] == 'ConvNeXt'
         else (0.55, 0.45) if class_strength[c] == 'B3'
         else (0.4, 0.6)
      for c in range(17)}),
]

# Val에서 검증
for label, weights in configs:
    ens = np.zeros_like(val_probs['efficientnet_b3'])
    for c in range(17):
        w_b3, w_conv = weights[c]
        ens[:, c] = w_b3 * val_probs['efficientnet_b3'][:, c] + w_conv * val_probs['convnext_small_fb_in22k_ft_in1k'][:, c]
    preds = np.argmax(ens, axis=1)
    f1 = f1_score(val_labels, preds, average='macro')
    print(f'  {label:<35} Val F1: {f1:.4f}')

# 가장 좋은 config로 test 제출 파일 생성
best_label, best_weights = max(configs, key=lambda x: f1_score(
    val_labels, np.argmax(sum(x[1][c][0] * val_probs['efficientnet_b3'][:, c:c+1] +
    x[1][c][1] * val_probs['convnext_small_fb_in22k_ft_in1k'][:, c:c+1]
    for c in range(17)), axis=1), average='macro'))

# test probs에 적용
ens_test = np.zeros_like(b3_probs)
for c in range(17):
    w_b3, w_conv = best_weights[c]
    ens_test[:, c] = w_b3 * b3_probs[:, c] + w_conv * conv_probs[:, c]

preds_test = np.argmax(ens_test, axis=1)
diff = (preds_test != np.argmax(0.4 * b3_probs + 0.6 * conv_probs, axis=1)).sum()

df_test['target'] = preds_test
df_test.to_csv(OUTPUT_DIR / 'submission_classweight_ensemble.csv', index=False)
print(f'\n저장: submission_classweight_ensemble.csv | 40:60 대비 {diff}장 차이')

print(f'   최적 config: {best_label}')
 # 클래스별 가중치 — Val에서 효과 없음 (0장 차이), 전역 40:60이 최적

In [ ]:
import numpy as np
import pandas as pd
from config import *

# test 예측 분석
b3_probs = np.load(OUTPUT_DIR / 'probs_efficientnet_b3_tta10.npy')
conv_probs = np.load(OUTPUT_DIR / 'probs_convnext_small_fb_in22k_ft_in1k_tta10.npy')
ens_probs = 0.4 * b3_probs + 0.6 * conv_probs
ens_preds = np.argmax(ens_probs, axis=1)
ens_conf = ens_probs.max(axis=1)

# 클래스 3, 7 예측 현황
for c in [3, 7]:
    mask = ens_preds == c
    print(f'\n클래스 {c}로 예측된 샘플: {mask.sum()}장')
    print(f'  평균 confidence: {ens_conf[mask].mean():.3f}')
    print(f'  confidence < 0.7: {(ens_conf[mask] < 0.7).sum()}장')
    print(f'  confidence < 0.5: {(ens_conf[mask] < 0.5).sum()}장')
    
    # 2순위 예측 분석
    top2 = np.argsort(ens_probs[mask], axis=1)[:, -2]
    print(f'  2순위 예측 분포: {dict(zip(*np.unique(top2, return_counts=True)))}')

# 클래스 3↔7 경계에 있는 샘플
mask_37 = (ens_preds == 3) | (ens_preds == 7)
probs_3 = ens_probs[mask_37, 3]
probs_7 = ens_probs[mask_37, 7]
margin = np.abs(probs_3 - probs_7)
print(f'\n클래스 3/7 예측 중 margin < 0.2 (불확실): {(margin < 0.2).sum()}장')
print(f'클래스 3/7 예측 중 margin < 0.1 (매우 불확실): {(margin < 0.1).sum()}장')

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import StratifiedShuffleSplit
from augmentation import get_train_transforms, get_valid_transforms
from train import build_model
from PIL import Image
import numpy as np
import pandas as pd
from config import *

# ============================================
# 1) 클래스 3, 7 전문가 모델 학습
# ============================================
df_train = pd.read_csv(TRAIN_CSV)
df_37 = df_train[df_train['target'].isin([3, 7])].copy()
df_37['binary_target'] = (df_37['target'] == 7).astype(int)  # 0=class3, 1=class7
print(f'클래스 3: {(df_37["binary_target"]==0).sum()}장 | 클래스 7: {(df_37["binary_target"]==1).sum()}장')

# 9:1 split
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
tr_idx, val_idx = next(sss.split(df_37, df_37['binary_target']))
df_37_tr = df_37.iloc[tr_idx]
df_37_val = df_37.iloc[val_idx]

class BinaryDocDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = np.array(Image.open(self.img_dir / row['ID']).convert('RGB'))
        img = self.transform(image=img)['image']
        return img, row['binary_target']

# 작은 모델로 빠르게 학습 (EfficientNet-B0)
model_expert = build_model('efficientnet_b0', 2, True).to('cuda')

train_ds = BinaryDocDataset(df_37_tr, TRAIN_DIR, get_train_transforms(384, version='v3'))
val_ds = BinaryDocDataset(df_37_val, TRAIN_DIR, get_valid_transforms(384))
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

optimizer = torch.optim.AdamW(model_expert.parameters(), lr=5e-5, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20, eta_min=1e-6)
criterion = nn.CrossEntropyLoss()

best_val_acc = 0
for epoch in range(20):
    model_expert.train()
    for images, labels in train_loader:
        images, labels = images.to('cuda'), labels.to('cuda')
        outputs = model_expert(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    scheduler.step()
    
    # Val
    model_expert.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            outputs = model_expert(images.to('cuda'))
            correct += (outputs.argmax(1).cpu() == labels).sum().item()
            total += len(labels)
    val_acc = correct / total
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model_expert.state_dict(), OUTPUT_DIR / 'expert_3v7.pth')
    if (epoch + 1) % 5 == 0:
        print(f'  Epoch {epoch+1:2d} | Val Acc: {val_acc:.4f} (best: {best_val_acc:.4f})')

print(f'\n전문가 모델 학습 완료 | Best Val Acc: {best_val_acc:.4f}')

# ============================================
# 2) 불확실 샘플에 전문가 모델 적용
# ============================================
model_expert.load_state_dict(torch.load(OUTPUT_DIR / 'expert_3v7.pth', map_location='cuda'))
model_expert.eval()

# 기존 앙상블 probs
b3_probs = np.load(OUTPUT_DIR / 'probs_efficientnet_b3_tta10.npy')
conv_probs = np.load(OUTPUT_DIR / 'probs_convnext_small_fb_in22k_ft_in1k_tta10.npy')
ens_probs = 0.4 * b3_probs + 0.6 * conv_probs
ens_preds = np.argmax(ens_probs, axis=1)

# margin < 0.2인 불확실 샘플 찾기
df_test = pd.read_csv(SAMPLE_CSV)
mask_37 = (ens_preds == 3) | (ens_preds == 7)
probs_3 = ens_probs[:, 3]
probs_7 = ens_probs[:, 7]
margin = np.abs(probs_3 - probs_7)
uncertain_mask = mask_37 & (margin < 0.2)
uncertain_indices = np.where(uncertain_mask)[0]
print(f'\n🔍 불확실 샘플: {len(uncertain_indices)}장')

# 전문가 모델로 재판단 (TTA 적용)
from augmentation import get_tta_transforms
tta_transforms = get_tta_transforms(384)

expert_probs = np.zeros((len(uncertain_indices), 2))
for tta_t in tta_transforms:
    ds = BinaryDocDataset(
        df_test.iloc[uncertain_indices].assign(binary_target=0),
        TEST_DIR, tta_t)
    loader = DataLoader(ds, batch_size=16, shuffle=False, num_workers=2)
    batch_probs = []
    with torch.no_grad():
        for images, _ in loader:
            outputs = model_expert(images.to('cuda'))
            batch_probs.append(torch.softmax(outputs, dim=1).cpu().numpy())
    expert_probs += np.concatenate(batch_probs, axis=0)
expert_probs /= len(tta_transforms)

expert_preds = np.argmax(expert_probs, axis=1)  # 0=class3, 1=class7
expert_classes = np.where(expert_preds == 0, 3, 7)

# 기존 예측과 비교
original_preds = ens_preds[uncertain_indices]
flipped = (expert_classes != original_preds).sum()
print(f'  전문가 모델이 뒤집은 예측: {flipped}장 / {len(uncertain_indices)}장')

# 최종 예측 생성
final_preds = ens_preds.copy()
final_preds[uncertain_indices] = expert_classes

total_changed = (final_preds != ens_preds).sum()
df_test['target'] = final_preds
df_test.to_csv(OUTPUT_DIR / 'submission_expert_ensemble.csv', index=False)
print(f'\n저장: submission_expert_ensemble.csv | 40:60 대비 {total_changed}장 변경')

del model_expert; torch.cuda.empty_cache()

# 전문가 모델 — Val Acc 0.70으로 낮아 미사용